<a href="https://colab.research.google.com/github/ashok-bisht/Context-Aware_Misinformation_Detection_ML_and_Gen_AI/blob/main/notebooks/1.0-eda-data-cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 1. EDA, Load and Merge

* Load and read df_true and df_fake data.
* View the description of the true and fake sets.
* Label for the true and fake sets (1, 0).

## Clean the data:

* Concat the two datasets into df.
* Remove unused columns, keeping only the title, text and label.
* Remove missing rows with drop null.
* Remove extra spaces.
* Check if the number of real/fake records is equal.
* Mix the data to ensure training.
* Calculate the length of each title in a data point.
* Remove the publisher info like Reuter, CNN etc from the text from both dataset.


In [ ]:
#Initialize folder paths
import os
import re
import pandas as pd

# 1. Define the base directory path on your Google Drive
base_drive_folder = "/content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection"

# Define explicit input, temp and output folder paths
input_folder_path = os.path.join (base_drive_folder, "input")
temp_folder_path = os.path.join (base_drive_folder, "temp")
clean_folder_path = os.path.join (base_drive_folder, "clean")

input_file_true = os.path.join(input_folder_path, "True.csv")
input_file_fake = os.path.join(input_folder_path, "Fake.csv")
input_temp_true = os.path.join(temp_folder_path, "True_Pub_Cleaned.csv")
input_temp_fake = os.path.join(temp_folder_path, "Fake_Pub_Cleaned.csv")


## clean datasets by removing the publisher info

In [ ]:

# Define the regex cleaning function
def strip_journalism_fingerprints(text, publishers_to_scrub=None):
    if not isinstance(text, str):
        return ""

    # 1. Clean up spacing and weird characters/newlines at the start
    text = text.strip()
    text = text.replace("\u00a0", " ")  # Fix NBSP

    # Standardize punctuation
    punctuation_map = {
        "’": "'", "‘": "'",  # Curly apostrophes -> Straight apostrophe
        "“": '"', "”": '"',  # Curly quotes -> Straight quotes
        "–": "-", "—": "-",  # En/Em dashes -> Standard hyphen
    }
    for curly, straight in punctuation_map.items():
        text = text.replace(curly, straight)
    text = text.replace("â€™", "'")      # Fix broken UTF-8 apostrophes ("â€™" -> "'")

    # Fix "isn t" -> "isn't", "don t" -> "don't"
    text = re.sub(r"\b(isn|don|didn|doesn|can|wasn|weren|haven|hasn|hadn|won|wouldn|shouldn|couldn|aren)\s+t\b", r"\1't", text, flags=re.IGNORECASE)

    # Fix spaces around real apostrophes like "don ' t" or "don 't"
    text = re.sub(r"\b(don|didn|doesn|isn|can)\s*'\s*t\b", r"\1't", text, flags=re.IGNORECASE)



    # 2. Remove leading bracketed corrections/clarifications at the very start
    # Matches: "(In 2nd paragraph...)"
    leading_bracket_pattern = r"^\([^)]+\)\s*"
    text = re.sub(leading_bracket_pattern, "", text)

    # 3. Strip standard datelines (e.g., "NEW YORK (Reuters) - ")
    dateline_pattern = r"^[A-Z\s,]+(?:\s\([^)]+\))?\s*[\s\-\-–—]\s*"
    text = re.sub(dateline_pattern, "", text)

    # 4. Remove standalone bracketed publisher or author markers (e.g., "(Reuters)" or "By Terray Sylvester")
    # This specifically target patterns like "(Reuters)" or "By John Doe (Reuters)" left over in the text
    byline_pattern = r"(?:By\s+[A-Za-z\s]+)?\s*\([^)]+\)"
    text = re.sub(byline_pattern, "", text)

    # 5. Scrub specific publisher words anywhere else in the text
    if publishers_to_scrub:
        escaped_publishers = [re.escape(pub) for pub in publishers_to_scrub]
        scrub_pattern = r"\b(" + "|".join(escaped_publishers) + r")\b"
        text = re.sub(scrub_pattern, "XYZ", text, flags=re.IGNORECASE)

    # Final cleanup of double spaces left behind by removals
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# ============================================================
# EXECUTION PIPELINE
# ============================================================
try:

    #******************************************************
    # Process True file
    #******************************************************

    # Load the original Kaggle CSV file
    print(f"🔄 Loading dataset True from: {input_file_true}")
    df_true= pd.read_csv(input_file_true, encoding='utf-8')
    print("✂️ Stripping datelines and publisher signatures from text for True...")

    # Apply the regex function exclusively to the text column
    publishers = ["Reuters", "CNN", "Associated Press", "BBC"]
    df_true["text"] = df_true["text"].apply(lambda x: strip_journalism_fingerprints(x, publishers_to_scrub=publishers))
    # On title column
    df_true["title"] = df_true["title"].apply(lambda x: strip_journalism_fingerprints(x, publishers_to_scrub=publishers))

    # Save to the new destination file name
    print(f"💾 Saving cleaned True file to: {temp_folder_path}")
    df_true.to_csv(input_temp_true, index=False)

    #******************************************************
    # Process Fake file
    #******************************************************

    print(f"🔄 Loading dataset Fake from: {input_file_fake}")
    # Load the original Kaggle CSV file
    df_fake= pd.read_csv(input_file_fake, encoding='utf-8')

    print("✂️ Stripping datelines and publisher signatures from text for True...")

    # Apply the regex function exclusively to the text column
    df_fake["text"] = df_fake["text"].apply(lambda x: strip_journalism_fingerprints(x, publishers_to_scrub=publishers))
    # on title column
    df_fake["title"] = df_fake["title"].apply(lambda x: strip_journalism_fingerprints(x, publishers_to_scrub=publishers))

    # Save to the new destination file name
    print(f"💾 Saving cleaned Fake file to: {temp_folder_path}")
    df_fake.to_csv(input_temp_fake, index=False)

    print("✅ Process complete! The data leakage loophole has been fixed.")

except FileNotFoundError:
    print(
        f"❌ Error: Could not find 'True.csv' at '{input_folder_path}'."
        " Please make sure your Google Drive is mounted using "
        " 'from google.colab import drive; drive.mount(\"/content/drive\")'"
    )
except Exception as e:
    print(f"❌ An error occurred during processing: {str(e)}")


## Change the column name text -> content and merge dataset


In [ ]:
import numpy as np # linear algebra
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Add a 'label' column (0 for fake, 1 for true)
df_fake['label'] = 0
df_true['label'] = 1

# Rename 'text' column to 'content' for consistency
df_fake.rename(columns={'text': 'content'}, inplace=True)
df_true.rename(columns={'text': 'content'}, inplace=True)

# Concatenate the dataframes
df_combined = pd.concat([df_fake, df_true], ignore_index=True)

# Display the first few rows and info of the combined dataframe
print("Combined DataFrame Head:")
display(df_combined.head())
print("\nCombined DataFrame Info:")
df_combined.info()
df_combined.describe()

In [ ]:
#print label==0, subject wise counts
print ("Fake News:")
print(df_combined[df_combined['label'] == 0]['subject'].value_counts())
print ("--"*50)
print ("True News:")
print(df_combined[df_combined['label'] == 1]['subject'].value_counts())
print (df_combined.shape)

In [ ]:
#Remove redundant columns

df_combined.drop(['date','subject'], axis=1, inplace=True)
print (df_combined.shape)

In [ ]:
#Drop NA records
df_combined.dropna(inplace=True)
print ("After Drop NA",df_combined.shape)
#Trim spaces at end
df_combined['title'] = df_combined['title'].astype(str).str.strip()
df_combined['content'] = df_combined['content'].astype(str).str.strip()
print ("After Trim Space", df_combined.shape)
#remove the duplicate records
df_combined.drop_duplicates(inplace=True)
print ("After Duplicate removal", df_combined.shape)


In [ ]:
#check true vs fake
print ("Check the distribution of True and False")
print(df_combined["label"].value_counts())

In [ ]:
# Randomly mix the data
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df_combined.head()

# Temp - update the Title with XYZ and check if results are better


In [ ]:
#update titel from df_combined to "XYZ"
# df_combined['title'] = "XYZ"
# df_combined.head()

## Save Cleaned Dataset to the google drive Path


In [ ]:
# Save the df_combined to this location
import os

# 1. Define the Google Drive folder and file name
combined_csv_path = os.path.join(clean_folder_path, 'combined_news.csv')

# 2. Ensure the directory exists
os.makedirs(clean_folder_path, exist_ok=True)

print("Saving DataFrame to Google Drive... Please wait.")

# 3. Save to CSV (index=False prevents pandas from adding an extra row numbers column)
df_combined.to_csv(combined_csv_path, index=False)

print(f"🎉 Success! Dataset successfully saved to: {combined_csv_path}")



In [ ]:
# Install spaCy and download the English model
!pip install spacy
!python -m spacy download en_core_web_sm

### 2. Text Cleaning with spaCy

Now, clean the text data using spaCy. This involves:
-   **Loading the spaCy model**: `en_core_web_sm`.
-   **Tokenization**: Breaking down text into individual words.
-   **Stopword Removal**: Removing common words that don't add much meaning.
-   **Lowercasing**: Converting all text to lowercase.
-   **Removing Punctuation and Special Characters**.
-   **Lemmatization**: Reducing words to their base form.

After cleaning, save the processed DataFrame to a new CSV file named `cleaned_news_data.csv`.

In [ ]:
# run if the colab fails in middle.
#load the DF from the drive
# import pandas as pd
# #pd.set_option('display.max_colwidth', 50)
# df_combined = pd.read_csv(combined_csv_path)
# df_combined.head()

In [ ]:
import os
from datetime import datetime
import pandas as pd
import spacy

# ==========================================
# ⚙️ CONFIGURATION BLOCK
# ==========================================
START_ROW = 0     # Change this to resume (e.g., 15000) if it fails midway
N = 1000          # Configurable 'n' rows per batch, CSV append, and log update
# ==========================================

# 1. Setup Folder and File Paths
spacy_csv_file = os.path.join(clean_folder_path, 'cleaned_news_spacy.csv')
if os.path.exists(spacy_csv_file):
    os.remove(spacy_csv_file)
    print(f"Successfully deleted existing old file {spacy_csv_file}")

# 2. Load spaCy with optimized disabling to maximize performance
nlp = spacy.load('en_core_web_sm', disable=['tok2vec', 'parser', 'ner'])

# 5. Generate Log File with Unique Timestamp Suffix
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f"progress_log_{current_time}.txt"
log_filepath = os.path.join(clean_folder_path, log_filename)

# Reusable helper function to process a list of texts through nlp.pipe
def clean_text_list(text_list):
    cleaned = []
    for doc in nlp.pipe(text_list, batch_size=250, n_process=-1):
        tokens = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop and not token.pos_ == "PROPN"]
        cleaned.append(" ".join(tokens))
    return cleaned

total_rows = len(df_combined)
print(f"Starting processing from row {START_ROW} out of {total_rows} total rows (Batch size N = {N})...")



# 3. Process and Save in Configurable Batches
for start in range(START_ROW, total_rows, N):
    end = min(start + N, total_rows)

    # Slice the current chunk from the main DataFrame
    batch_df = df_combined.iloc[start:end].copy()

    # --- PROCESS COLUMN 1: TITLE ---
    title_texts = batch_df['title'].fillna("").astype(str).str.lower().tolist()
    batch_df['cleaned_title'] = clean_text_list(title_texts)

    # --- PROCESS COLUMN 2: CONTENT ---
    content_texts = batch_df['content'].fillna("").astype(str).str.lower().tolist()
    batch_df['cleaned_content'] = clean_text_list(content_texts)

    # 4. Save to CSV in Append Mode
    if start == 0 and not os.path.exists(spacy_csv_file):
        batch_df.to_csv(spacy_csv_file, index=False, mode='w')
    else:
        # Append mode ('a') bypasses writing the column headers again
        batch_df.to_csv(spacy_csv_file, index=False, mode='a', header=False)

    log_content = (
        f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
        f"Status: Success\n"
        f"Last Completed Index: {end - 1}\n"
        f"Processed Rows: {start} to {end - 1}\n"
        f"Total Data Progress: {end}/{total_rows} rows complete.\n"
    )

    with open(log_filepath, "a") as log_file:
        log_file.write(log_content)

    print(f"✅ Appended rows {start} to {end-1} (Title, Content cleaned). Log: {log_filename}")

print(f"\n🎉 All processing completed! Final file saved with multiple clean features at: {spacy_csv_file}")


### 3. TF-IDF Vectorization

Finally, perform TF-IDF (Term Frequency-Inverse Document Frequency) vectorization on the `cleaned_content` column. TF-IDF is a numerical statistic that reflects how important a word is to a document in a collection or corpus.

I will use `TfidfVectorizer` from `sklearn.feature_extraction.text` to convert the text data into a matrix of TF-IDF features. The resulting TF-IDF matrix will be saved as a new CSV file named `tfidf_vectors.csv` for use by other team members.

In [ ]:
import os
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse
import numpy as np

# 1. Load the cleaned data
df_spacy = pd.read_csv(spacy_csv_file)

# 2. Critical Step: Fill empty/missing cells with blank strings to prevent crashes
df_spacy['cleaned_title'] = df_spacy['cleaned_title'].fillna("")
df_spacy['cleaned_content'] = df_spacy['cleaned_content'].fillna("")

# 3. Initialize separate vectorizers using a ColumnTransformer
# We apply different max_feature caps based on typical text length per field
preprocessor = ColumnTransformer(
    transformers=[
        ('title_tfidf', TfidfVectorizer(max_features=5000), 'cleaned_title'),
        ('content_tfidf', TfidfVectorizer(max_features=25000), 'cleaned_content')
    ]
)

print("Vectorizing features in parallel...")
# 4. Transform your columns into a horizontal combined sparse matrix
tfidf_matrix = preprocessor.fit_transform(df_spacy)

# 5. Extract unique, descriptive names for all newly generated feature columns
feature_names = preprocessor.get_feature_names_out()

# Define output file paths

sparse_matrix_file_path = os.path.join(base_drive_folder, 'tfidf_sparse_matrix.npz')
feature_names_file_path = os.path.join(base_drive_folder, 'tfidf_feature_names.npy')
labels_file_path = os.path.join(base_drive_folder, 'tfidf_labels.csv')

# 6. Save the sparse matrix using scipy.sparse.save_npz
scipy.sparse.save_npz(sparse_matrix_file_path, tfidf_matrix)
print(f"\nSparse TF-IDF matrix successfully saved to: {sparse_matrix_file_path}")

# 7. Save feature names (as a numpy array)
np.save(feature_names_file_path, feature_names)
print(f"TF-IDF feature names successfully saved to: {feature_names_file_path}")

# 8. Save the classification labels separately
df_spacy['label'].to_csv(labels_file_path, index=False)
print(f"Classification labels successfully saved to: {labels_file_path}")
